# E7 — Phân tích số đo đã lưu; dedicated GPU measurements DEFERRED

Không đo GPU hoặc chạy model inference tại máy cá nhân. Chỉ đọc Kaggle efficiency/history logs. Không nâng trạng thái E7 thành COMPLETE vì có latency nhúng trong trainer.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/master_session1.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.master_session1 import ARCHIVE_NAME, ARCHIVE_SHA256, BUNDLE_HASH, analyze
TABLES = ROOT / "results/tables/master_session1"
FULL_AUDIT = False  # True needs the private original ZIP and matching dataset; never trains.
REBUILD_FIGURES = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_MASTER_ARCHIVE", str(ROOT / "artifacts/master_session1" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, resamples=10000)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["E1_success"] == 16 and audit["E2_success"] == 6 and audit["E7_gpu_state"] == "DEFERRED"
print("Source SHA256:", ARCHIVE_SHA256)
print("Mode:", "raw ZIP re-audited" if FULL_AUDIT else "versioned audited aggregate tables; not a fresh raw re-audit")


In [ ]:
runs = pd.read_csv(TABLES / "runs.csv")
display(runs[["experiment", "model", "variant", "seed", "training_seconds", "clustering_seconds", "validation_seconds", "checkpoint_seconds", "test_evaluation_seconds", "inference_mean_ms", "inference_p95_ms", "inference_scope", "peak_gpu_memory_mib", "measurement_device", "gpu_name"]])


Clustering nằm trong training time. Latency batch 1 gồm encoding/scoring toàn catalogue, không sorting/disk I/O; 5 warm-up và 20 repeats. Baseline CPU fit không trực tiếp tương đương neural GPU training. Memory là peak allocated, không phải tổng VRAM. Component sums không phải session elapsed time.


In [ ]:
budget = pd.read_csv(TABLES / "remaining_budget_scenarios.csv")
display(budget)
print("Planning scenario hours (not measured remaining runs):", budget.max_epoch_mean_rate_scenario_seconds.sum()/3600)
assert not budget.remaining_run_actually_executed.any()


In [ ]:
from gen_fig_master_session1 import generate
if REBUILD_FIGURES:
    generate(ROOT)
display(Image(filename=str(ROOT / "results/figures/master_session1/recorded_efficiency.png")))


## Báo cáo và trạng thái

Đọc [báo cáo tiếng Việt](../docs/MASTER_SESSION1_ANALYSIS_VI.md). Phạm vi thu gọn do tài nguyên GPU và yêu cầu môn học, không theo mô hình có test score cao. E1 complete, E2 partial, E3 pending/reuse available, E4/E5 deferred; TIGER/CRAMER theory only. Không tuyên bố các phase chưa chạy đã hoàn tất.
